# Wave-maker tool: a guided tour

This notebook walks through the `wavemaker` package one step at a time. Each section explains what a part of the code does, why, and then runs it so you can see the result.

The package follows the HR Wallingford workflow:

```
 wave you want          →  what the paddle must do        →  checks        →  file for the PLC
 (H, T, h  or  Hs, Tp)     (dispersion, transfer function,    (stroke,          (x in mm, one value
                            time series)                       breaking, ...)    per sample)
```

**Contents**
1. Setup
2. Dispersion: how long is the wave?
3. Transfer function: how far must the paddle move?
4. A regular (monochromatic) wave
5. Spectra: describing an irregular sea
6. An irregular wave
7. Safety and validity checks
8. Cross-tank sloshing modes
9. Froude scaling
10. Projects: saving cases and outputs
11. The operating envelope of the tank
12. Running the test suite

Run the cells in order (Shift + Enter). Every cell is safe: nothing here talks to the PLC.

## 1. Setup

This notebook lives in `notebooks/` inside the repository. The first cell adds the repository root to Python's search path, so `import wavemaker` works without installing the package. If you have run `pip install -e .` already, the cell does no harm.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

import wavemaker
print("wavemaker version", wavemaker.__version__, "loaded from", Path(wavemaker.__file__).parent)

### The facility settings

`Facility` (in `config.py`) holds everything the code knows about the tank and paddle: tank size, the 600 mm stroke, the working depth limit, the sample interval and so on.

Values we have **not** yet confirmed are stored as `None`. The code never guesses them. Instead, any safety check that needs one of these values reports *unconfigured* and refuses to let the signal go to the paddle. When Kriben sends the drive limits, you type them into the project's `facility.json` and those checks come alive.

In [ ]:
from wavemaker.config import Facility

fac = Facility()
for name, value in vars(fac).items():
    print(f"{name:24s} {value}")
print("\nStill to confirm:", fac.unconfigured())

## 2. Dispersion: how long is the wave?

A wave's length L depends on its period T **and** on the water depth h. The link is the linear dispersion relation:

$$\omega^2 = g\,k\,\tanh(kh), \qquad \omega = \frac{2\pi}{T}, \quad k = \frac{2\pi}{L}$$

You cannot rearrange this for k directly, so `wavenumber()` solves it by **Newton iteration**: it makes a good first guess, then corrects the guess repeatedly until the answer stops changing (to about 15 significant figures). It usually needs three or four steps.

Helper functions build on it:

| Function | Gives you |
|---|---|
| `wavenumber(omega, h)` | k (rad/m) |
| `wavelength(T, h)` | L (m) |
| `period_from_wavelength(L, h)` | T (s) for a chosen wavelength |
| `celerity(T, h)` | wave speed C = L/T (m/s) |
| `group_velocity(T, h)` | speed at which wave energy travels (m/s) |

All take numbers or NumPy arrays, in SI units.

In [ ]:
from wavemaker.dispersion import wavenumber, wavelength, period_from_wavelength, celerity, group_velocity

T, h = 1.2, 0.5
k = wavenumber(2*np.pi/T, h)
print(f"T = {T} s in h = {h} m:")
print(f"  k  = {k:.4f} rad/m")
print(f"  L  = {wavelength(T, h):.4f} m")
print(f"  kh = {k*h:.4f}")
print(f"  C  = {celerity(T, h):.3f} m/s,  Cg = {group_velocity(T, h):.3f} m/s")

# Check: plug k back into the dispersion relation. The two sides should agree.
g = 9.81
print(f"\nomega^2 = {(2*np.pi/T)**2:.12f}")
print(f"g k tanh(kh) = {g*k*np.tanh(k*h):.12f}")

**Worked example: the spreadsheet error.** The old `waveDesign.xlsx` gave T = 2.04 s for a 2 m wave in 0.3 m of water. Dispersion says otherwise:

In [ ]:
T_correct = period_from_wavelength(2.0, 0.3)
print(f"L = 2 m, h = 0.3 m  ->  T = {T_correct:.3f} s   (spreadsheet gave 2.04 s)")
print(f"Round trip: wavelength(T, 0.3) = {wavelength(T_correct, 0.3):.6f} m")

**Deep, intermediate and shallow water.** The plot shows L against T for three depths. For short periods all depths agree: the wave doesn't "feel" the bottom (deep water, L = gT²/2π). For long periods the wave is squeezed by the depth (shallow water, L ≈ T√(gh)). Most tank tests sit in between, which is why we solve the full relation.

In [ ]:
T = np.linspace(0.4, 4, 300)
for h in (0.3, 0.45, 0.6):
    plt.plot(T, wavelength(T, h), label=f"h = {h} m")
plt.plot(T, 9.81*T**2/(2*np.pi), "k--", lw=1, label="deep-water limit")
plt.ylim(0, 10); plt.xlabel("T (s)"); plt.ylabel("L (m)"); plt.legend()
plt.title("Wavelength against period");

## 3. Transfer function: how far must the paddle move?

The **transfer function** tells you how tall a wave you get for a given paddle stroke:

$$\text{TF} = \frac{H}{S} = \frac{\text{wave height}}{\text{paddle stroke}}$$

It depends only on kh. For our **piston** paddle (Biesel theory):

$$\frac{H}{S} = \frac{2(\cosh 2kh - 1)}{\sinh 2kh + 2kh}$$

To make a wave of height H, the paddle must travel S = H / TF. So:

- **Short waves (large kh):** TF → 2. A 10 mm stroke makes a 20 mm wave. Easy.
- **Long waves (small kh):** TF → kh. The paddle must travel much further than the wave height. Long waves are what use up the stroke.

`transfer.py` also has the **flap** formula, in case the paddle is ever changed. The code rewrites both formulas slightly so they don't lose accuracy at small kh or overflow at large kh; the maths is identical.

In [ ]:
from wavemaker.transfer import piston_tf, flap_tf

kh = np.linspace(0.01, 4, 400)
plt.plot(kh, piston_tf(kh), label="piston (ours)")
plt.plot(kh, flap_tf(kh), label="flap")
plt.axhline(2, color="k", ls="--", lw=1)
plt.xlabel("kh"); plt.ylabel("H / S"); plt.legend()
plt.title("Wavemaker transfer functions");

In [ ]:
# Stroke needed for a 50 mm wave at a few periods, h = 0.5 m
h, H = 0.5, 0.05
print(" T (s)    kh     H/S    stroke S (mm)")
for T in (0.8, 1.2, 1.6, 2.0, 3.0):
    kh = wavenumber(2*np.pi/T, h) * h
    tf = piston_tf(kh)
    print(f"{T:5.1f}  {kh:6.3f}  {tf:6.3f}   {H/tf*1000:8.1f}")

## 4. A regular (monochromatic) wave

`monochromatic(H, T, h, dt)` builds the paddle's position, sample by sample:

$$x(t) = \frac{S}{2}\sin(\omega t)$$

Two details matter for the real machine:

- **Ramps.** Starting a paddle at full swing would jolt the drive and send a sharp front down the tank. So the signal fades in and out with a smooth **cosine ramp** (5 periods by default). The first and last samples are exactly zero: the paddle starts and finishes at its centre.
- **Target surface elevation.** Alongside x(t), the function returns η(t), the wave the theory predicts at the paddle. We use it for checks and for comparing with gauges later.

The result is a `WaveSeries` object with these parts:

| Attribute | Meaning |
|---|---|
| `t` | time (s) |
| `x` | paddle displacement (m), positive toward the beach |
| `eta` | target water surface (m) |
| `dt` | sample interval (s) |
| `steady` | the part of the record between the ramps |
| `meta` | a dictionary of everything else: k, kh, L, TF, stroke, ... |

In [ ]:
from wavemaker.signals import monochromatic

s = monochromatic(H=0.05, T=1.2, h=0.5, dt=0.01, n_steady=10, n_ramp=5)

for key in ("L", "kh", "tf", "stroke_S", "duration_s"):
    print(f"{key:12s} {s.meta[key]:.4f}")

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
ax[0].plot(s.t, s.x*1000); ax[0].set_ylabel("paddle x (mm)")
ax[1].plot(s.t, s.eta*1000, color="C1"); ax[1].set_ylabel("target η (mm)"); ax[1].set_xlabel("t (s)")
for a in ax:
    a.axvspan(s.t[0], s.t[s.steady.start], color="grey", alpha=0.2)
    a.axvspan(s.t[s.steady.stop], s.t[-1], color="grey", alpha=0.2)
ax[0].set_title("Grey bands are the ramps");

Notice that the peaks of η come a quarter period *before* the peaks of x. That is correct: the water is pushed hardest when the paddle moves fastest, which is as it passes the centre.

## 5. Spectra: describing an irregular sea

A real sea is a mix of many waves of different periods. A **spectrum** S(f) says how much energy sits at each frequency. Two numbers summarise it:

- **Hs** (strictly Hm0): the significant wave height, 4 × the standard deviation of the water surface.
- **Tp**: the peak period, where the spectrum is highest.

`spectra.py` offers three shapes:

| Kind | Use |
|---|---|
| `"pm"` Pierson–Moskowitz | a fully developed sea |
| `"jonswap"` | a growing sea with a sharper peak; γ controls how sharp (3.3 is typical) |
| `"csv"` | your own spectrum from a file with columns f, S |

`SpectrumSpec` bundles the choice. Every spectrum is scaled so that its area equals Hs²/16, which is what makes Hs come out right.

In [ ]:
from wavemaker.spectra import SpectrumSpec

f = np.linspace(0.01, 2.5, 1000)
Hs, Tp = 0.04, 1.5
for kind, gamma in [("pm", 1.0), ("jonswap", 3.3), ("jonswap", 7.0)]:
    spec = SpectrumSpec(kind, Hs, Tp, gamma)
    S = spec.density(f)
    area = np.trapezoid(S, f)
    plt.plot(f, S, label=f"{kind}, γ = {gamma}   (4√area = {4*np.sqrt(area)*1000:.1f} mm)")
plt.axvline(1/Tp, color="k", ls=":", lw=1)
plt.xlabel("f (Hz)"); plt.ylabel("S (m²/Hz)"); plt.legend()
plt.title("Same Hs and Tp, different shapes");

All three curves hold the same total energy (Hs = 40 mm). A larger γ piles more of it near the peak.

## 6. An irregular wave

`irregular(spec, h, dt, ...)` turns a spectrum into a paddle signal. In plain terms:

1. **Chop the spectrum into thin slices** between 0.5 fp and 3 fp (the *generation band*). Each slice becomes one sine wave. The slice width is 1 / (record length), so a 20-minute record gives slices 0.00083 Hz wide; the example below has about 2 000 components.
2. **Give each sine wave an amplitude** from the energy in its slice: a = √(2 S Δf).
3. **Give each a random phase**, so the waves don't all peak together. The random numbers come from a **seed**. Same seed, same sea, every time. The seed is always stored, so any test can be repeated exactly.
4. **Convert each wave amplitude to a paddle amplitude** by dividing by that component's transfer function.
5. **Add all the sine waves up** in one go with an inverse FFT, which is very fast.
6. **Remove any offset and apply ramps.**
7. **Check the result**: measure Hs and Tp of the target η and compare with what was asked for.

Two amplitude modes exist:
- `"deterministic"` (default): each component gets exactly its share of energy. Hs lands on target every time.
- `"random_complex"`: amplitudes also vary randomly, as in nature. Hs then scatters by about 1 % from run to run.

The signal repeats only after the full record, so no pattern repeats during a test.

In [ ]:
from wavemaker.signals import irregular

spec = SpectrumSpec("jonswap", hs=0.04, tp=1.5, gamma=3.3)
si = irregular(spec, h=0.5, dt=0.01, seed=7)

m = si.meta
print(f"Record length      {m['duration_s']:.0f} s  ({si.x.size} samples)")
print(f"Band               {m['f_min']:.3f} – {m['f_max']:.3f} Hz, {m['n_components']} components")
print(f"Seed               {m['seed']}")
print(f"Target  Hs, Tp     {m['Hs']:.4f} m, {m['Tp']:.3f} s")
print(f"Realised Hm0, Tp   {m['Hm0_realised']:.4f} m, {m['Tp_realised']:.3f} s")
print(f"Peak paddle travel {np.max(np.abs(si.x))*1000:.1f} mm")

In [ ]:
from wavemaker.plotting import plot_series
plot_series(si, window_s=120);   # first 2 minutes of the time series, plus the spectrum

The bottom panel is the proof that the method works: the spectrum measured from the generated η (blue) sits on the target (dashed).

**Repeatability.** Run it again with the same seed and you get the identical signal. Change the seed and you get a different sea with the same statistics.

In [ ]:
a = irregular(spec, 0.5, 0.01, seed=7)
b = irregular(spec, 0.5, 0.01, seed=7)
c = irregular(spec, 0.5, 0.01, seed=8)
print("Same seed identical?     ", np.array_equal(a.x, b.x))
print("Different seed identical?", np.array_equal(a.x, c.x))
print(f"Hm0 with seed 8:           {c.meta['Hm0_realised']:.4f} m")

r = irregular(spec, 0.5, 0.01, amplitude_mode="random_complex", seed=7)
print(f"Random-complex Hm0:        {r.meta['Hm0_realised']:.4f} m  (varies a little by chance)")

## 7. Safety and validity checks

`run_checks(series, facility)` inspects a generated signal before it can go anywhere near the paddle. Each check returns one of four results:

| Result | Meaning | Blocks upload? |
|---|---|---|
| `pass` | fine | no |
| `warn` | allowed, but read the message | no |
| `fail` | unsafe or outside the theory | **yes** |
| `unconfigured` | the limit it needs isn't known yet | **yes** |

What gets checked:

| Check | Question it answers |
|---|---|
| Water depth | Is h within the 0.6 m working limit? |
| Stroke | Does the paddle stay inside ±300 mm, less a safety margin? |
| Velocity, acceleration | Can the drive actually move this fast? (worked out from x(t)) |
| Depth-limited breaking | Is H/h below 0.78? Taller waves break. |
| Steepness breaking (Miche) | Is the wave too steep for its length? |
| Individual-wave breaking | (irregular only) How many single waves in the record would break? |
| Freeboard | Will crests stay below the tank rim? |
| Shallow-water range | kh < 0.3: first-order theory makes unwanted long waves. |
| Ursell number | Is the wave so nonlinear that linear theory fails? |
| Cross-tank modes | Is the frequency near a sloshing mode across the 1.2 m width? (Section 8) |
| Sampling | Are there enough samples per wave for a smooth signal? |
| Realised Hm0, Tp | (irregular only) Did the synthesis hit the target within 2 %? |

`upload_allowed(results)` is True only if nothing fails and nothing is unconfigured.

**With today's facility settings every signal is blocked,** because the drive limits aren't known yet. That is intended.

In [ ]:
from wavemaker.checks import run_checks, upload_allowed

results = run_checks(si, Facility())
for r in results:
    print(r)
print("\nUpload allowed:", upload_allowed(results))

To see the checks working end to end, the next cell uses **made-up test limits**. They are *not* drive data. Replace them with Festo's figures when they arrive.

In [ ]:
test_fac = Facility(stroke_margin_m=0.02, freeboard_margin_m=0.10,
                    max_velocity_m_s=1.0, max_acceleration_m_s2=10.0)   # PLACEHOLDERS for demonstration only

def report(series, fac, title):
    res = run_checks(series, fac)
    print(f"--- {title}")
    for r in res:
        if r.status != "pass":
            print(r)
    print("Upload allowed:", upload_allowed(res), "\n")

report(monochromatic(0.03, 1.5, 0.5, 0.01), test_fac, "Small, comfortable wave")
report(monochromatic(0.18, 4.0, 0.3, 0.01), test_fac, "Long wave in shallow water (runs out of stroke)")
report(monochromatic(0.12, 0.7, 0.5, 0.01), test_fac, "Short, tall wave (too steep)")

## 8. Cross-tank sloshing modes

Water can slosh *across* the tank as well as travel along it. The 1.2 m width has natural sloshing frequencies:

$$k_m = \frac{m\pi}{B}, \qquad \omega_m^2 = g\,k_m \tanh(k_m h)$$

Two things can excite them:
- running the paddle **at** a sloshing frequency, if anything in the tank is slightly asymmetric;
- running it at **twice** a sloshing frequency, which can grow *cross-waves* at half the paddle frequency.

The tool warns within ±5 % of either. The plot shows where the first modes fall as depth changes; avoid choosing test periods on these lines.

In [ ]:
from wavemaker.checks import transverse_mode_frequencies

depths = np.linspace(0.3, 0.6, 50)
fm = np.array([transverse_mode_frequencies(1.2, h, 9.81, n=2) for h in depths])
plt.plot(depths, 1/fm[:, 0], label="mode 1 (paddle at this period)")
plt.plot(depths, 1/fm[:, 1], label="mode 2")
plt.plot(depths, 1/(2*fm[:, 0]), "--", label="half mode-1 period (cross-waves)")
plt.xlabel("water depth h (m)"); plt.ylabel("paddle period to avoid (s)"); plt.legend()
plt.title("Periods to avoid in a 1.2 m wide tank");
print("At h = 0.5 m: mode 1 at T = %.2f s, cross-wave risk at T = %.2f s" %
      (1/transverse_mode_frequencies(1.2, 0.5, 9.81)[0], 1/(2*transverse_mode_frequencies(1.2, 0.5, 9.81)[0])))

## 9. Froude scaling

To model a real sea in the tank, scale it by a length ratio λ (prototype / model). Under Froude similarity:

- lengths (H, h, L) divide by λ;
- times (T) divide by √λ.

`FroudeScale(lam)` does the arithmetic both ways. (The old spreadsheet used λ^0.75 for time by mistake; this is the corrected version.)

In [ ]:
from wavemaker.scaling import FroudeScale

fs = FroudeScale(25)
print(fs.describe())
H_m, T_m, h_m = fs.to_model(H=2.0, T=10.0, h=12.0)
print(f"Prototype H = 2 m, T = 10 s, h = 12 m  ->  model H = {H_m:.3f} m, T = {T_m:.2f} s, h = {h_m:.2f} m")

# The spreadsheet check: lambda = 10, model T = 2.04 s
print(f"lambda = 10, model T = 2.04 s -> prototype T = {FroudeScale(10).to_prototype(0.1, 2.04, 0.5)[1]:.2f} s (sheet: 11.47 s)")

## 10. Projects: saving cases and outputs

A **project** is a folder that keeps everything for one set of experiments together:

```
my_project/
    project.json          name, date, notes
    facility.json         tank and paddle settings (edit the [TBC] values here)
    cases/<case>.json     each wave case you define
    series/<case>.csv         t, x, η in metres
    series/<case>_plc.csv     x in millimetres, ready for the PLC
    series/<case>_checks.txt  the check report
    run_log.jsonl         a diary of everything done, with times
```

Define a case with `RegularCase` or `SpectrumCase`, save it, then call `generate()`. If an irregular case had no seed, the seed actually used is written back into the case file, so the run can always be repeated.

The metre-to-millimetre conversion happens in one place only, `units.py`, so there's no chance of a unit mix-up elsewhere.

The cell below makes a throwaway project in a temporary folder.

In [ ]:
import tempfile, json
from wavemaker.project import Project, RegularCase, SpectrumCase

root = Path(tempfile.mkdtemp()) / "demo_project"
proj = Project.create(root, name="Notebook demo", notes="Throwaway test")

proj.save_case(RegularCase(name="reg_H05_T12", H=0.05, T=1.2, h=0.5))
proj.save_case(SpectrumCase(name="js_Hs04_Tp15", h=0.5, spectrum="jonswap", Hs=0.04, Tp=1.5))
print("Cases:", proj.list_cases())

series, results = proj.generate("js_Hs04_Tp15")
print("Upload allowed:", upload_allowed(results))
print("Seed written back to case:", proj.load_case("js_Hs04_Tp15").seed)

print("\nFiles in the project:")
for p in sorted(root.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(root))

In [ ]:
# What the PLC file looks like: a short header, then one position (mm) per line
print((root / "series" / "js_Hs04_Tp15_plc.csv").read_text()[:200])
print("... last log entry:")
print(json.dumps(json.loads((root / "run_log.jsonl").read_text().splitlines()[-1]), indent=1)[:600])

## 11. The operating envelope of the tank

Putting the pieces together: for each period, what is the tallest regular wave the tank can make? Three things cap it:

- **stroke**: H ≤ TF × (usable stroke);
- **steepness breaking**: H ≤ 0.142 L tanh(kh);
- **depth breaking**: H ≤ 0.78 h.

The usable stroke below assumes a 20 mm margin (a placeholder). The velocity and acceleration limits of the drive will add two more curves once Kriben supplies them; expect them to cut off the short, tall waves on the left.

In [ ]:
h = 0.5
half_stroke = 0.300 - 0.020                 # placeholder 20 mm margin
T = np.linspace(0.5, 4.0, 300)
k = wavenumber(2*np.pi/T, h); kh = k*h; L = 2*np.pi/k

H_stroke = piston_tf(kh) * 2*half_stroke
H_steep  = 0.142 * L * np.tanh(kh)
H_depth  = np.full_like(T, 0.78*h)
H_max    = np.minimum.reduce([H_stroke, H_steep, H_depth])

plt.plot(T, H_stroke, "--", label="stroke limit")
plt.plot(T, H_steep,  "--", label="steepness (Miche)")
plt.plot(T, H_depth,  "--", label="depth breaking")
plt.fill_between(T, 0, H_max, alpha=0.2, label="achievable")
plt.ylim(0, 0.45); plt.xlabel("T (s)"); plt.ylabel("H (m)"); plt.legend()
plt.title(f"Regular-wave envelope, h = {h} m (before drive speed limits)");

In practice, freeboard, nonlinearity and reflections cap useful heights well below these curves. Treat this plot as an outer bound, not a target.

## 12. Running the test suite

The `tests/` folder holds 31 automatic tests. They check the code against published tables, hand calculations and known limits (for example, the dispersion solver against Dean & Dalrymple's tables, and the two spreadsheet errors). Run them after any change to the code. All should pass.

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(repo_root / "tests")],
                     capture_output=True, text=True, cwd=repo_root)
print(out.stdout[-1500:] or out.stderr[-1500:])

## Try it yourself

Change the numbers below and re-run the cell. Try a wave near 1.33 s (a cross-tank mode at h = 0.5 m), a long wave in shallow water, or a steep short wave, and watch which checks respond.

In [ ]:
H, T, h = 0.06, 1.33, 0.5
s = monochromatic(H, T, h, dt=0.01)
report(s, test_fac, f"H = {H} m, T = {T} s, h = {h} m")
print(f"Stroke needed: {s.meta['stroke_S']*1000:.1f} mm of 600 mm")